In [ ]:
import os
import cv2
import random
import numpy as np
from sklearn.model_selection import train_test_split


In [ ]:
categories = ['with_mask', 'without_mask']

In [ ]:
data = []

for category in categories:
    path = os.path.join("data", category) # this "data" --> dataset folder name
    label = categories.index(category)

    for file in os.listdir(path):

        if file.startswith("."):
            continue

        img_path = os.path.join(path, file)
        img = cv2.imread(img_path)

        if img is None:
            print("Skipping:", img_path)
            continue

        img = cv2.resize(img, (224, 224))
        data.append([img, label])

print("Total images loaded:", len(data))

In [ ]:
random.shuffle(data)

In [ ]:
X = []
Y = []

for features, label in data:
    X.append(features)
    Y.append(label)


X = np.array(X)
Y = np.array(Y)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

print("Training images:", len(X_train))
print("Testing images:", len(X_test))

In [ ]:
# Normalize image pixel values from [0, 255] to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [ ]:
from keras.applications.vgg16 import VGG16

vgg = VGG16()

In [ ]:
from keras import Sequential
from keras.layers import Dense

model = Sequential()

for layer in vgg.layers[:-1]:
    model.add(layer)

for layer in model.layers:
    layer.trainable = False

model.add(Dense(1, activation='sigmoid'))


In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer='Adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
model.fit(
    X_train,
    Y_train,
    epochs=5,
    validation_data=(X_test, Y_test)
)

model.save("model.keras")
print("Model saved successfully!")